<a href="https://colab.research.google.com/github/rachana1707-S/Adaptive-MedRAG/blob/feature%2Fneeraj-generation/notebooks/03_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# install the Hugging Face datasets library to download PubMedQA
!pip install -q datasets

In [2]:
from datasets import load_dataset

pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
print(pubmedqa)

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})


In [3]:
# Looking at the first example so we can see what each field looks like
labeled_data = pubmedqa["train"]
first_example = labeled_data[0]

print("pubid:", first_example["pubid"])
print("question:", first_example["question"])
print("final_decision:", first_example["final_decision"])
print()
print("context keys:", first_example["context"].keys())
print()
print("long_answer:", first_example["long_answer"])

pubid: 21645374
question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
final_decision: yes

context keys: dict_keys(['contexts', 'labels', 'meshes', 'reasoning_required_pred', 'reasoning_free_pred'])

long_answer: Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of CsA in a whole plant system. Overall, our findings implicate the mitochondria as playing a critical and early role in developmentally regulated PCD in the lace plant.


In [4]:
# the context field holds the abstract, split into sections
abstract_sections = first_example["context"]["contexts"]
section_labels = first_example["context"]["labels"]

for section_index in range(len(abstract_sections)):
    print(f"[{section_labels[section_index]}]")
    print(abstract_sections[section_index])
    print()

[BACKGROUND]
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.

[RESULTS]
The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (LPCD). Window stage leaves were stained with the mitochondrial dye M

In [5]:
from collections import Counter

# count how many yes/no/maybe answers there are
label_counts = Counter(labeled_data["final_decision"])

total_examples = len(labeled_data)
for label, count in label_counts.items():
    percentage = (count / total_examples) * 100
    print(f"{label}: {count} ({percentage:.1f}%)")

yes: 552 (55.2%)
no: 338 (33.8%)
maybe: 110 (11.0%)


In [6]:
# find out roughly how long each abstract is in a word.
abstract_word_counts = []

for example in labeled_data:
    abstract_sections = example["context"]["contexts"]
    full_abstract = " ".join(abstract_sections)

    words = full_abstract.split()
    word_count = len(words)
    abstract_word_counts.append(word_count)

shortest = min(abstract_word_counts)
longest = max(abstract_word_counts)
average = round(sum(abstract_word_counts) / len(abstract_word_counts), 1)

print("shortest abstract:", shortest, "words")
print("longest abstract:", longest, "words")
print("average abstract:", average, "words")

shortest abstract: 49 words
longest abstract: 398 words
average abstract: 200.2 words


##Part 2: LLM Setup

In [7]:
import torch

#checking if colab actually is working on GPU
gpu_is_available = torch.cuda.is_available()
print("GPU is available:", gpu_is_available)
print("GPU name:", torch.cuda.get_device_name(0))

GPU is available: True
GPU name: Tesla T4


In [8]:
!pip install -q -U transformers accelerate

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# load the model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
)

print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on: cuda:0


In [10]:
MAX_NEW_TOKENS = 20
SYSTEM_MESSAGE = "You are a helpful biomedical research assistant."

def ask_model(user_message):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_message},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # turn the prompt text into token ids and move them onto the GPU
    model_inputs = tokenizer(prompt_text, return_tensors="pt")
    input_ids = model_inputs["input_ids"].to(model.device)
    attention_mask = model_inputs["attention_mask"].to(model.device)

    # generate the answer
    output_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )

    prompt_length = input_ids.shape[1]
    all_tokens = output_ids[0]
    answer_tokens = all_tokens[prompt_length:]

    # turn the answer tokens back into readable text
    answer_text = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    return answer_text

In [11]:
import time

for example_index in range(3):
    example = labeled_data[example_index]
    question = example["question"]

    user_message = question + "\nAnswer with one word: yes, no, or maybe."

    # time how long the model takes to answer
    start_time = time.time()
    model_answer = ask_model(user_message)
    end_time = time.time()

    response_time_ms = round((end_time - start_time) * 1000)

    print("Question:", question)
    print("Model answer:", model_answer)
    print("Gold answer:", example["final_decision"])
    print("Time:", response_time_ms, "ms")
    print("-" * 50)

Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Model answer: Yes
Gold answer: yes
Time: 1665 ms
--------------------------------------------------
Question: Landolt C and snellen e acuity: differences in strabismus amblyopia?
Model answer: Yes
Gold answer: no
Time: 78 ms
--------------------------------------------------
Question: Syncope during bathing in infants, a pediatric form of water-induced urticaria?
Model answer: Yes
Gold answer: yes
Time: 96 ms
--------------------------------------------------
